# Прототип агентной системы: Суммаризатор учебника (Textbook Summarizer)
**Тип архитектуры:** Условная оркестрация с роутером (Conditional Router)  
**Используемый API провайдер:** OpenRouter API

Выполнено в рамках практической работы по проектированию интеллектуальных мультиагентных систем.

## 1. Блок-схема архитектуры пайплайна
Ниже приведена Mermaid-схема, описывающая поток данных, маршрутизацию и вызовы внешнего API языковой модели:

```mermaid
graph TD
    A[Ввод пользователя] --> B[Маршрутизатор: Классификатор запроса]
    B -->|'definitions'| C[Узел Definitions: Извлечь только словарь]
    B -->|'summary'| D[Узел Summary: Написать общий конспект]
    B -->|'qa'| E[Узел QA: Ответить на точечный вопрос]
    C --> F[Сборка ответа и вывод]
    D --> F
    E --> F
    B -.->|Классификация LLM| LLM[LLM API]
    C -.->|Вызов LLM| LLM
    D -.->|Вызов LLM| LLM
    E -.->|Вызов LLM| LLM
```

*Примечание: Вы можете скопировать этот код схемы для отрисовки в draw.io, Mermaid Live Editor или просмотреть прямо в GitHub Markdown.*

## 2. Установка зависимостей
Для запуска пайплайна нам понадобятся библиотеки LangChain, LangGraph или официальный SDK Google GenAI.

In [ ]:
!pip install langchain-openai langchain-core langgraph -qq

## 3. Инициализация и подключение API модели
Настраиваем подключение к провайдеру LLM API. Не забудьте вписать ваши авторизационные ключи.

In [ ]:
# Подключение к OpenRouter API (например, модель deepseek-chat или gpt-4o-mini)
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = "sk-or-v1-..." # Вставьте ваш API-ключ OpenRouter
OPENROUTER_MODEL = "deepseek/deepseek-chat" # Использование доступной модели

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7
)

## 4. Реализация логики агентов и пайплайна
Определяем состояние системы (State) и узлы (Nodes) графа/цепочки.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 1. Определяем состояние агентов
class RouterState(TypedDict):
    input_text: str
    category: str
    answer: str

# 2. Определение узлов
def router_node(state: RouterState):
    print("🤖 Узел 1: Классификация запроса...")
    prompt = f"Определи категорию запроса для темы summarizer (только одно слово в нижнем регистре):\nЗапрос: {state['input_text']}"
    category = llm.invoke(prompt).content.strip().lower()
    return {"category": category}

def expert_node_1(state: RouterState):
    print("➡️ Перенаправлено на Эксперта 1 (Технические детали / Факты)")
    prompt = f"Дай строгий технический ответ: {state['input_text']}"
    return {"answer": llm.invoke(prompt).content}

def expert_node_2(state: RouterState):
    print("➡️ Перенаправлено на Эксперта 2 (Форматирование / Пересказ / Стиль)")
    prompt = f"Дай развернутый обучающий структурированный ответ: {state['input_text']}"
    return {"answer": llm.invoke(prompt).content}

def general_expert_node(state: RouterState):
    print("➡️ Перенаправлено на Универсального Эксперта")
    prompt = f"Дай полезный практичный ответ: {state['input_text']}"
    return {"answer": llm.invoke(prompt).content}

# Функция условных переходов
def route_decision(state: RouterState) -> Literal["node1", "node2", "general"]:
    cat = state["category"]
    if "definitions" in cat or "syntax" in cat or "action" in cat:
        return "node1"
    elif "summary" in cat or "style" in cat or "comedy" in cat:
        return "node2"
    else:
        return "general"

# 3. Построение условного графа
workflow = StateGraph(RouterState)
workflow.add_node("classify", router_node)
workflow.add_node("expert1", expert_node_1)
workflow.add_node("expert2", expert_node_2)
workflow.add_node("general", general_expert_node)

workflow.add_edge(START, "classify")
workflow.add_conditional_edges("classify", route_decision, {
    "node1": "expert1",
    "node2": "expert2",
    "general": "general"
})
workflow.add_edge("expert1", END)
workflow.add_edge("expert2", END)
workflow.add_edge("general", END)

app = workflow.compile()

def run_pipeline(text):
    result = app.invoke({
        "input_text": text,
        "category": "",
        "answer": ""
    })
    return result["answer"]

## 5. Запуск примеров запросов (Демонстрация работы)
Запускаем систему на 2–3 разноплановых примерах с выводом логов и результатов работы.

In [ ]:
# Примеры тестовых запросов для демонстрации работы пайплайна
tests = [
    "Глава 3. Эволюционная теория Ч. Дарвина.\nОсновной движущей силой эволюции видов Чарльз Дарвин назвал естественный отбор. Естественный отбор — это процесс, в результате которого организмы, более приспособленные к своей среде обитания, выживают и оставляют потомство.\nДругим ключевым понятием является наследственная изменчивость. Изменчивость обеспечивает появление новых признаков у особей одного вида, а наследственность позволяет передавать эти адаптивные признаки следующим поколениям. Таким образом, в популяции накапливаются полезные мутации.\nДарвин также выделил борьбу за существование как предпосылку отбора. Из-за ограниченности ресурсов (пищи, территории, партнеров) особи вступают в конкуренцию, где побеждают наиболее приспособленные.",
    "Выпиши только самые главные определения и термины по этой теме.",
    "Составь краткий тест из 3 вопросов по прочитанному материалу для проверки знаний."
]

for i, test_input in enumerate(tests, 1):
    print(f"\n" + "="*60)
    print(f"ТЕСТОВЫЙ ЗАПРОС №{i}:")
    print(test_input[:150] + "..." if len(test_input) > 150 else test_input)
    print("-"*60)
    
    try:
        # Запуск пайплайна
        
        
        output = run_pipeline(test_input)
        
        print("🤖 ФИНАЛЬНЫЙ ОТВЕТ АГЕНТА:")
        print(output)
    except Exception as e:
        print(f"❌ Ошибка выполнения: {e}")


## 6. Критерии оценки (Чек-лист)
✅ Корректная блок-схема, отражающая структуру агентов и потоков данных в пайплайне  
✅ Код корректно запускается, демонстрирует агентное взаимодействие на основе выбранного туториала  
✅ Успешное подключение выбранного API  
✅ Воспроизводятся 3 примера выдачи (логи/результаты видны в выдачах в ipynb)  
✅ Решение выложено с комментариями, описанием шагов и блок-схемой